# 03 · Baseline logistic regression with 5-fold cross-validation

One validation split (notebook 02) gives one number, and we cannot tell how much of it is luck. Here the same baseline model is scored with **5-fold patient-level cross-validation**:

- **80%** of Hospital A's patients (development set) are split into 5 folds. Each fold is the validation set once, and the model is trained on the other 4.
- **20%** are the test set. They are set aside and not used anywhere in this notebook.
- Everything that learns from the data (dropping near-empty features, medians, scaling, the model) is re-fitted inside each fold, on that fold's training patients only.

Two models are compared on the same folds: **unweighted** logistic regression, and **weighted** (`class_weight="balanced"`), which counts every septic hour as much as all non-septic hours together. Only about 2% of hours are septic, so this choice matters. All later notebooks use the weighted model.

In [1]:
import pandas as pd

import sepsis as s

data_A = s.load_hospital("A")

# 80% development (cross-validation) / 20% test (set aside)
dev_ids, test_ids = s.split_patients(data_A, test_size=0.20)
fold_of = s.make_folds(data_A, dev_ids)

status = s.patient_status(data_A)
print(f"Development: {len(dev_ids):,} patients ({int(status.loc[dev_ids].sum())} septic)")
print(f"Test (not used here): {len(test_ids):,} patients ({int(status.loc[test_ids].sum())} septic)")
print(fold_of.value_counts().sort_index().rename("patients per fold").to_string())

Development: 16,268 patients (1432 septic)
Test (not used here): 4,068 patients (358 septic)
1    3254
2    3254
3    3254
4    3253
5    3253


In [2]:
from pathlib import Path
f = s.data_dir() / "training_setA" / "p016172.psv"
print(f.stat().st_size, "bytes")
print(open(f).read()[:200])

8811 bytes
HR|O2Sat|Temp|SBP|MAP|DBP|Resp|EtCO2|BaseExcess|HCO3|FiO2|pH|PaCO2|SaO2|AST|BUN|Alkalinephos|Calcium|Chloride|Creatinine|Bilirubin_direct|Glucose|Lactate|Magnesium|Phosphate|Potassium|Bilirubin_total|


In [3]:
# Forward fill once for all development patients (it only uses each patient's own past).
# No missingness features: this is the plain baseline.
dev = s.prepare_dev(data_A, dev_ids, fold_of, missingness=False)

oofs = {
    "unweighted": s.run_cv(dev, s.ALL_FEATURES, class_weight=None),
    "weighted": s.run_cv(dev, s.ALL_FEATURES),                       # class_weight="balanced" is the default
}

trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features


## 1. AUROC and AUPRC

These do not depend on a cutoff: they only ask whether septic hours get higher probabilities than non-septic ones. The second table is the difference *weighted − unweighted* in each fold. Both models are scored on the same folds, so a difference that has the same sign in every fold is real, while one smaller than its std is probably noise.

In [4]:
summary, diffs = s.compare_auc(oofs)
display(summary.round(4))
display(diffs.round(4))

AUROC          AUPRC        
              mean    std    mean     std
Features                                 
unweighted  0.7411  0.021  0.0842  0.0141
weighted    0.7462  0.023  0.0827  0.0142

,AUROC,AUPRC
Features,weighted,weighted
1,0.0105,-0.0009
2,0.0037,-0.0022
3,0.0072,-0.0002
4,0.0081,-0.0020
5,-0.0040,-0.0019
mean,0.0051,-0.0015
std,0.0056,0.0009


## 2. What happens at the default cutoff of 0.5

Weighting mostly moves the probabilities up for everybody. With so few septic hours, the unweighted model rarely reaches 0.5, so at that cutoff it raises almost no alarms.

In [5]:
rows = []
for name, oof in oofs.items():
    pred = (oof["y_prob"].to_numpy() >= 0.5).astype(int)
    utility = s.make_utility_scorer(oof["PatientID"], oof["y_true"], oof["Hour"])
    rows.append({"Model": name, **s.threshold_metrics(oof["y_true"], pred),
                 "Clinical utility": utility(pred), "Alarms (positive hours)": int(pred.sum())})

pd.DataFrame(rows).set_index("Model").round(4)

,F1,Precision,Recall,FPR,Clinical utility,Alarms (positive hours)
Model,,,,,,
unweighted,0.0095,0.1204,0.005,0.0008,0.0040,565
weighted,0.0949,0.0513,0.626,0.2569,0.2995,167183


## 3. Each model at its own best cutoff

Comparing both at 0.5 is unfair to the unweighted model. Here each one gets the cutoff (0.10 to 0.95) with the highest clinical utility on the pooled out-of-fold predictions, and the last two columns show how much the utility moves between folds at that cutoff.

In [6]:
s.best_cutoff_table(oofs).round(4)

,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
unweighted,0.10,0.1353,0.1520,0.1219,0.0151,0.1050,0.1050,0.0122
weighted,0.55,0.1094,0.0608,0.5454,0.1871,0.3099,0.3098,0.0379


## Conclusion

The weighted baseline reaches an AUROC of about 0.75 and an AUPRC of about 0.08 across the 5 folds, with a best clinical utility of about 0.31 at cutoff 0.55. Compare the unweighted row with it: if AUROC barely changes but the 0.5-cutoff results do, weighting mainly shifts *where* the cutoff should be, not how well the model ranks patients. The 20% test set and Hospital B stay untouched until the final evaluation. The next notebooks keep this weighted model and only change the input features.